# Imports

In [6]:
import pandas as pd
import os
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# Configs

In [7]:
df_enade = pd.read_csv("../enade_ready_to_use/enade_pós-process_18-23.csv", encoding="utf-8-sig", low_memory=False)

df_municipios = pd.read_csv("../enade_ready_to_use/municipios.csv", encoding="utf-8-sig", low_memory=False)
df_cursos = pd.read_csv("../enade_ready_to_use/cursos.csv", encoding="utf-8-sig", low_memory=False)




In [15]:
df_enade.columns

Index(['ano_enade', 'id_curso', 'categoria_administrativa',
       'modalidade_graduacao', 'turno_graduacao', 'id_municipio', 'sexo',
       'idade', 'cor_raca', 'ano_fim_ensino_medio', 'ano_ingresso_graduacao',
       'tipo_escola_ensino_medio', 'primeira_geracao', 'escolaridade_pai',
       'escolaridade_mae', 'motivacao_curso', 'renda_familiar',
       'horas_trabalho', 'cotas', 'nota_geral', 'nota_formacao_geral',
       'nota_componentes_especifico', 'avaliacao_dificuldade_fg',
       'avaliacao_dificuldade_ce', 'avaliacao_equipamentos',
       'avaliacao_ambiente'],
      dtype='str')

In [8]:
df_desnormalizado = df_enade.merge(df_municipios, on="id_municipio", how="left").merge(df_cursos, on="id_curso", how="left")

In [9]:
df_desnormalizado.groupby('id_curso')['id_municipio'].nunique().max()

np.int64(726)

In [10]:
ordem = [
    "id_curso",
    "nome_curso",
    "categoria_administrativa",
    "modalidade_graduacao",
    "turno_graduacao",
    
    "ano_enade",
    
    
    "id_municipio",
    "nome_municipio",
    "uf",
    "regiao",

    "sexo",
    "idade",
    "cor_raca",

    "ano_fim_ensino_medio",
    "ano_ingresso_graduacao",
    "tipo_escola_ensino_medio",
    "primeira_geracao",
    "escolaridade_pai",
    "escolaridade_mae",
    "motivacao_curso",
    "renda_familiar",
    "horas_trabalho",
    "cotas",

    "nota_geral",
    "nota_formacao_geral",
    "nota_componentes_especifico",

    "avaliacao_dificuldade_fg",
    "avaliacao_dificuldade_ce",
    "avaliacao_equipamentos",
    "avaliacao_ambiente"

    
]

In [11]:
df_desnormalizado = df_desnormalizado[ordem]

In [12]:
df_desnormalizado.insert(0, "id_participacao", range(1, len(df_desnormalizado) + 1))

In [13]:
df_desnormalizado.shape

(2472225, 31)

In [9]:
print(f"valores nulos por coluna:")
print(df_desnormalizado.isnull().mean() * 100)

valores nulos por coluna:
id_participacao                 0.000000
id_curso                        0.000000
nome_curso                      0.000000
categoria_administrativa        1.236497
modalidade_graduacao            0.000000
turno_graduacao                 0.993720
ano_enade                       0.000000
id_municipio                    0.000000
nome_municipio                  0.000000
uf                              0.000000
regiao                          0.000000
sexo                            0.033249
idade                           0.033371
cor_raca                       13.233666
ano_fim_ensino_medio            0.046598
ano_ingresso_graduacao          0.047730
tipo_escola_ensino_medio       13.234313
primeira_geracao               13.234515
escolaridade_pai               13.233828
escolaridade_mae               13.233666
motivacao_curso                13.235486
renda_familiar                 13.233828
horas_trabalho                 13.233949
cotas                          

In [14]:
copia = df_desnormalizado.copy()

In [16]:
cols_desempenho = ["nota_geral", "nota_formacao_geral", "nota_componentes_especifico"]

cols_avaliacao = ["avaliacao_dificuldade_fg", "avaliacao_dificuldade_ce", "avaliacao_equipamentos", "avaliacao_ambiente"]

cols_perfil_estudante = [
    "sexo", "idade", "cor_raca", "ano_fim_ensino_medio", "ano_ingresso_graduacao", "tipo_escola_ensino_medio", "primeira_geracao", "escolaridade_pai", "escolaridade_mae", "motivacao_curso", "renda_familiar", "horas_trabalho", "cotas"]

cols_oferta = ['id_curso', 'id_municipio']

In [17]:
dim_desempenho = (copia[cols_desempenho].drop_duplicates().reset_index(drop=True))

dim_desempenho["id_desempenho"] = dim_desempenho.index + 1

dim_desempenho = dim_desempenho[["id_desempenho"] + cols_desempenho]

In [18]:
dim_avaliacao = (copia[cols_avaliacao].drop_duplicates().reset_index(drop=True))
dim_avaliacao["id_avaliacao"] = dim_avaliacao.index + 1
dim_avaliacao = dim_avaliacao[["id_avaliacao"] + cols_avaliacao]

In [19]:
dim_perfil_estudante = (copia[cols_perfil_estudante].drop_duplicates().reset_index(drop=True))
dim_perfil_estudante["id_perfil_estudante"] = dim_perfil_estudante.index + 1
dim_perfil_estudante = dim_perfil_estudante[["id_perfil_estudante"] + cols_perfil_estudante]

In [20]:
dim_oferta = (copia[cols_oferta].drop_duplicates().reset_index(drop=True))
dim_oferta["id_oferta"] = dim_oferta.index + 1
dim_oferta = dim_oferta[["id_oferta"] + cols_oferta]

In [21]:
dim_municipio = copia[["id_municipio", "nome_municipio", "uf"]].drop_duplicates().reset_index(drop=True)
dim_uf = copia[["uf", "regiao"]].drop_duplicates().reset_index(drop=True)
dim_curso = copia[["id_curso", "nome_curso", "categoria_administrativa", "modalidade_graduacao", "turno_graduacao"]].drop_duplicates().reset_index(drop=True)

In [22]:
copia = copia.merge(dim_desempenho, on=cols_desempenho, how="left").merge(dim_avaliacao, on=cols_avaliacao, how="left").merge(dim_perfil_estudante, on=cols_perfil_estudante, how="left").merge(dim_oferta, on=cols_oferta, how="left")

In [24]:
ordem_certa = [
    'id_participacao',
    'id_oferta',
    'id_curso', 
    'nome_curso',
    'categoria_administrativa',
    'modalidade_graduacao',
    'turno_graduacao',
    'ano_enade',
    'id_municipio',
    'nome_municipio', 
    'uf',
    'regiao',
    'id_perfil_estudante', 
    'sexo',
    'idade', 
    'cor_raca',
    'ano_fim_ensino_medio', 
    'ano_ingresso_graduacao',
    'tipo_escola_ensino_medio', 
    'primeira_geracao', 
    'escolaridade_pai',
    'escolaridade_mae', 
    'motivacao_curso', 
    'renda_familiar',
    'horas_trabalho', 
    'cotas',
    'id_desempenho', 
    'nota_geral', 
    'nota_formacao_geral',
    'nota_componentes_especifico',
    'id_avaliacao', 
    'avaliacao_dificuldade_fg',
    'avaliacao_dificuldade_ce', 
    'avaliacao_equipamentos',
    'avaliacao_ambiente'
    ]

copia = copia[ordem_certa]

In [27]:
copia.columns 

Index(['id_participacao', 'id_oferta', 'id_curso', 'nome_curso',
       'categoria_administrativa', 'modalidade_graduacao', 'turno_graduacao',
       'ano_enade', 'id_municipio', 'nome_municipio', 'uf', 'regiao',
       'id_perfil_estudante', 'sexo', 'idade', 'cor_raca',
       'ano_fim_ensino_medio', 'ano_ingresso_graduacao',
       'tipo_escola_ensino_medio', 'primeira_geracao', 'escolaridade_pai',
       'escolaridade_mae', 'motivacao_curso', 'renda_familiar',
       'horas_trabalho', 'cotas', 'id_desempenho', 'nota_geral',
       'nota_formacao_geral', 'nota_componentes_especifico', 'id_avaliacao',
       'avaliacao_dificuldade_fg', 'avaliacao_dificuldade_ce',
       'avaliacao_equipamentos', 'avaliacao_ambiente'],
      dtype='str')

In [ ]:
copia.to_csv("enade_desnormalizado.csv", index=False, encoding="utf-8-sig")

In [39]:
copia.shape

(2472225, 35)

In [29]:
dim_avaliacao.to_csv("dim_avaliacao.csv", index=False, encoding="utf-8-sig")
dim_desempenho.to_csv("dim_desempenho.csv", index=False, encoding="utf-8-sig")
dim_perfil_estudante.to_csv("dim_perfil_estudante.csv", index=False, encoding="utf-8-sig")
dim_municipio.to_csv("dim_municipio.csv", index=False, encoding="utf-8-sig")
dim_uf.to_csv("dim_uf.csv", index=False, encoding="utf-8-sig")
dim_curso.to_csv("dim_curso.csv", index=False, encoding="utf-8-sig")
dim_oferta.to_csv("dim_oferta.csv", index=False, encoding="utf-8-sig")


In [35]:
fato = copia[["id_participacao","id_oferta","id_perfil_estudante", "id_desempenho", "id_avaliacao"]]
fato.to_csv("fato.csv", index=False, encoding="utf-8-sig")

In [37]:
fato.shape

(2472225, 5)

In [32]:
print(f"valores nulos por coluna:")
print(copia.isnull().mean() * 100)

valores nulos por coluna:
id_participacao                 0.000000
id_oferta                       0.000000
id_curso                        0.000000
nome_curso                      0.000000
categoria_administrativa        1.236497
modalidade_graduacao            0.000000
turno_graduacao                 0.993720
ano_enade                       0.000000
id_municipio                    0.000000
nome_municipio                  0.000000
uf                              0.000000
regiao                          0.000000
id_perfil_estudante             0.000000
sexo                            0.033249
idade                           0.033371
cor_raca                       13.233666
ano_fim_ensino_medio            0.046598
ano_ingresso_graduacao          0.047730
tipo_escola_ensino_medio       13.234313
primeira_geracao               13.234515
escolaridade_pai               13.233828
escolaridade_mae               13.233666
motivacao_curso                13.235486
renda_familiar                 

In [33]:
head = copia.sample(frac=0.1).reset_index(drop=True)
head.head(5)



,id_participacao,id_oferta,id_curso,nome_curso,categoria_administrativa,modalidade_graduacao,turno_graduacao,ano_enade,id_municipio,nome_municipio,uf,regiao,id_perfil_estudante,sexo,idade,cor_raca,ano_fim_ensino_medio,ano_ingresso_graduacao,tipo_escola_ensino_medio,primeira_geracao,escolaridade_pai,escolaridade_mae,motivacao_curso,renda_familiar,horas_trabalho,cotas,id_desempenho,nota_geral,nota_formacao_geral,nota_componentes_especifico,id_avaliacao,avaliacao_dificuldade_fg,avaliacao_dificuldade_ce,avaliacao_equipamentos,avaliacao_ambiente
0,661246,4582,6208,Engenharia De Produção,Privada com fins lucrativos,Presencial,Integral,2019,3304557,RIO DE JANEIRO,RJ,Sudeste,2302,F,23.0,Branca,2009.0,2014.0,Todo em escola pública,Sim,Ensino Fundamental: 6º ao 9º ano,Ensino Fundamental: 6º ao 9º ano,Influência familiar,"De 1,5 a 3 salários mínimos)",Não estou trabalhando,Não,237345,35.0,42.8,32.4,155,Médio,Difícil,Concordo,Concordo
1,1125634,8746,2001,Pedagogia (Licenciatura),Privada com fins lucrativos,EAD,Noturno,2021,3106200,BELO HORIZONTE,MG,Sudeste,3620,F,24.0,Branca,2008.0,2019.0,Todo em escola pública,Sim,Ensino Fundamental: 1º ao 5º ano,Ensino Fundamental: 1º ao 5º ano,Valorização profissional,"De 1,5 a 3 salários mínimos)",Trabalho eventualmente,Não,55671,33.0,34.0,32.6,155,Médio,Difícil,Concordo,Concordo
2,1127920,8746,2001,Pedagogia (Licenciatura),Privada com fins lucrativos,EAD,Noturno,2021,3106200,BELO HORIZONTE,MG,Sudeste,3627,F,24.0,Branca,2008.0,2021.0,Todo em escola pública,Sim,Ensino Fundamental: 1º ao 5º ano,Ensino Fundamental: 1º ao 5º ano,Valorização profissional,"De 1,5 a 3 salários mínimos)",Trabalho eventualmente,Não,234886,33.2,21.4,37.1,117,Médio,Difícil,Concordo parcialmente,Concordo
3,792092,4282,5710,Engenharia Civil,Privada com fins lucrativos,Presencial,Integral,2019,3106200,BELO HORIZONTE,MG,Sudeste,2515,M,26.0,Preta,2013.0,2014.0,Todo em escola pública,Sim,Ensino Médio,Ensino Médio,Vocação,"De 3 a 4,5 salários mínimos)",Trabalho até 20 horas semanais,Não,130021,49.1,41.3,51.7,79,Médio,Médio,Concordo totalmente,Concordo totalmente
4,346589,32,38,Serviço Social,Privada com fins lucrativos,EAD,Matutino,2018,4113700,LONDRINA,PR,Sul,1319,M,29.0,Parda,2012.0,2014.0,Todo em escola privada,Não,Ensino Médio,Ensino Médio,Vocação,"De 4,5 a 6 salários mínimos)",Trabalho 40 horas semanais ou mais,Não,141849,51.7,57.5,49.8,149,Médio,Difícil,Concordo totalmente,Concordo totalmente


In [40]:
head.head(5).to_csv("head.csv", index=False, encoding="utf-8-sig")